# Uncertainty Quantification for Magnetic Field Prediction

This chapter explores advanced uncertainty quantification techniques for deep learning models in electromagnetic field prediction. Understanding uncertainty is crucial for reliable deployment in engineering applications where confidence in predictions directly impacts safety, regulatory compliance, and decision-making {cite}`amodei2016concrete,varshney2016engineering`.

**Chapter motivation**: Section 4d demonstrated that CNN predictions occasionally fail—particularly for geometric extrapolation and deep saturation conditions. Without uncertainty estimates, engineers cannot distinguish reliable predictions from unreliable ones, creating significant risk for automated design workflows {cite}`holm2020review,hulsebos2020experimental`.

## Learning Objectives

After completing this chapter, you will understand:

- **Sources of Uncertainty**: Aleatoric vs epistemic uncertainty in physics-based models {cite}`kendall2017uncertainties,hullermeier2021aleatoric`
- **Bayesian Neural Networks**: Probabilistic deep learning for uncertainty estimation {cite}`mackay1992bayesian,neal2012bayesian`
- **Monte Carlo Dropout**: Practical uncertainty quantification method {cite}`gal2016dropout,gal2015bayesian`
- **Deep Ensembles**: Model averaging for robust uncertainty estimates {cite}`lakshminarayanan2017simple`
- **Calibration and Validation**: Ensuring reliable uncertainty estimates {cite}`guo2017calibration,kuleshov2018accurate`

:::{seealso}
**Cross-references**:
- Section 4d: Performance analysis revealing when predictions fail
- Section 6: Physics-informed learning leveraging uncertainty for active data selection
:::

## Types of Uncertainty in Physics-Based Models

### 1. Aleatoric Uncertainty (Irreducible, Data-Dependent)

Aleatoric uncertainty arises from inherent randomness in observations, irreducible even with infinite training data {cite}`kendall2017uncertainties,hullermeier2021aleatoric`:

**Sources in electromagnetic modeling**:
- **Measurement noise**: Gaussmeter sensor accuracy (±0.1-0.5% typical)
- **FEA discretization**: Finite element mesh errors (0.1-0.5% for well-resolved meshes)
- **Material property variations**: B-H curve tolerances from manufacturing (±5-10% permeability variation)
- **Geometric tolerances**: Machining accuracy (±0.05-0.1 mm typical)

**Mathematical representation** {cite}`nix1994estimating`:

$$p(y|x, \theta) = \mathcal{N}(f(x, \theta), \sigma^2(x))$$

where $\sigma^2(x)$ is **input-dependent** (heteroscedastic noise)—higher near saturation regions, lower in linear operating regions.

### 2. Epistemic Uncertainty (Reducible, Model-Dependent)

Epistemic uncertainty stems from model limitations, reducible with more training data or improved architectures {cite}`kendall2017uncertainties,der2009aleatory`:

**Sources in electromagnetic modeling**:
- **Model capacity**: CNN with insufficient receptive field cannot capture long-range field interactions
- **Training data sparsity**: Undersampled regions of design space (e.g., only 50 samples at deep saturation)
- **Parameter estimation**: Non-convex loss landscape leading to suboptimal weights
- **Out-of-distribution**: Novel topologies (Halbach arrays) not seen in training

**Mathematical representation** via posterior distribution {cite}`mackay1992bayesian`:

$$p(y|x, \mathcal{D}) = \int p(y|x, \theta) p(\theta|\mathcal{D}) d\theta$$

where $p(\theta|\mathcal{D})$ represents uncertainty over network parameters given training data $\mathcal{D}$.

### 3. Total Predictive Uncertainty: Law of Total Variance

The total predictive uncertainty combines both sources {cite}`kiureghian2009aleatory,kendall2017uncertainties`:

$$\text{Var}[y|x, \mathcal{D}] = \underbrace{\mathbb{E}_{\theta}[\sigma^2(x, \theta)]}_{\text{Aleatoric}} + \underbrace{\text{Var}_{\theta}[\mu(x, \theta)]}_{\text{Epistemic}}$$

where:
- $\mu(x, \theta) = \mathbb{E}[y|x, \theta]$ = model prediction
- $\sigma^2(x, \theta)$ = predicted noise variance
- Expectation/variance taken over posterior $p(\theta|\mathcal{D})$

:::{important}
**Electromagnetic Design Implications**  

**Aleatoric uncertainty** {cite}`hullermeier2021aleatoric`:
- Cannot be reduced by more training data
- Represents fundamental limits of predictability
- **EM example**: Manufacturing tolerances always introduce ±2% torque variation

**Epistemic uncertainty** {cite}`kendall2017uncertainties`:
- Reducible by collecting more training samples in high-uncertainty regions
- **EM example**: High epistemic uncertainty at geometric boundaries → collect more FEA samples with chamfers/fillets
- **Active learning strategy**: Query FEA simulator at max(epistemic uncertainty) locations

**Design workflow**:
1. High epistemic uncertainty → model unreliable, collect more data
2. High aleatoric uncertainty → inherent noise, robust optimization needed
3. Both low → prediction reliable, proceed with design
:::

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Verify installation
print("Libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")



:::note
**Monte Carlo Dropout Requirements**  

This notebook uses PyTorch's dropout layers in a non-standard way:
- **Training mode during inference**: `model.train()` keeps dropout active
- **Multiple forward passes**: Sampling from approximate posterior
- **Statistical aggregation**: Mean and variance over stochastic predictions

This approach is mathematically equivalent to approximate variational inference
with Bernoulli distribution over network weights (Gal & Ghahramani, 2016).
:::

## Monte Carlo Dropout: Practical Uncertainty Quantification

### 1. Theoretical Foundation

Monte Carlo Dropout provides a computationally efficient approximation to Bayesian inference in deep neural networks {cite}`gal2016dropout,gal2015bayesian`.

**Key insight**: Dropout during inference samples from an approximate posterior over network weights, enabling uncertainty quantification without explicitly training a Bayesian neural network {cite}`gal2016uncertainty`.

**Mathematical foundation** {cite}`gal2016dropout`:

**Variational inference objective**: Minimize KL divergence between approximate posterior $q_{\theta}(w)$ and true posterior $p(w|\mathcal{D})$:

$$\mathcal{L}(\theta) = \text{KL}[q_{\theta}(w) || p(w)] - \mathbb{E}_{q_{\theta}(w)}[\log p(\mathcal{D}|w)]$$

**Dropout as approximate inference** {cite}`gal2015bayesian`:
- Each dropout mask samples a network configuration: $\tilde{w} \sim q_{\theta}(w)$
- Dropout probability $p$ induces a Bernoulli distribution over connections
- Multiple forward passes approximate the expectation: $\mathbb{E}_{q_{\theta}(w)}[f(x, w)]$

**Connection to Gaussian processes** {cite}`gal2015bayesian`: Deep networks with dropout at every weight layer correspond to an implicit Gaussian process approximation, providing a theoretical justification for uncertainty estimates.

### 2. Implementation Strategy

**Training phase** {cite}`srivastava2014dropout`:
- Standard dropout with probability $p$ (typically 0.1-0.5)
- Optimize weights using backpropagation and gradient descent
- No special modifications to loss function

**Inference phase** {cite}`gal2016dropout`:
- **Critical difference**: Keep dropout enabled (`model.train()` in PyTorch)
- Perform $T$ stochastic forward passes (typically T=50-100)
- Each pass samples a different sub-network via random dropout masks
- Aggregate predictions statistically

**Predictive statistics** {cite}`kendall2017uncertainties`:

**Mean prediction** (best estimate):
$$\hat{y}(x) = \frac{1}{T}\sum_{t=1}^{T} f_t(x, \tilde{w}_t)$$

**Predictive variance** (epistemic uncertainty):
$$\sigma^2(x) = \frac{1}{T}\sum_{t=1}^{T} f_t^2(x, \tilde{w}_t) - \hat{y}^2(x)$$

**Confidence intervals** (95% coverage):
$$\text{CI}_{95\%} = [\hat{y}(x) - 1.96\sigma(x), \; \hat{y}(x) + 1.96\sigma(x)]$$

:::{tip}
**Computational Efficiency for EM Field Prediction**  

**Advantages**:
- **Single model**: No need to train multiple networks (unlike ensembles)
- **Parallel inference**: T forward passes can run in parallel on GPU
- **Minimal overhead**: ~50-100× slower than single prediction, but still 1000× faster than FEA

**Typical performance for magnetic field prediction**:
- Single forward pass: 20-30 ms
- MC Dropout (T=50): 1-2 seconds per geometry
- Still enables design optimization with 100-1000 candidates (vs. days for FEA)

**Recommendation**: Use T=50 for interactive applications, T=100 for high-stakes decisions.
:::

In [ ]:
# Monte Carlo Dropout Implementation
class MCDropoutMagneticFieldNet(nn.Module):
    """
    Magnetic Field Prediction Network with Monte Carlo Dropout
    
    Implements Bayesian uncertainty quantification via dropout at inference time.
    Each forward pass samples a different sub-network, enabling estimation of
    epistemic uncertainty without training multiple models.
    
    Architecture:
    - Input: Geometric/material parameters (e.g., magnet dimensions, current density)
    - Hidden layers: Fully-connected with ReLU activation and dropout
    - Output: Magnetic field magnitude at a spatial location
    
    Reference: Gal & Ghahramani (2016) - Dropout as a Bayesian Approximation
    
    EM Application: Quantifies confidence in field predictions, especially useful
    for detecting when designs are outside training distribution (extrapolation).
    """

    def __init__(self, input_dim=3, hidden_dims=[64, 32, 16], output_dim=1, dropout_rate=0.2):
        """
        Initialize MC Dropout network
        
        Args:
            input_dim: Number of input features (geometry + material + excitation)
            hidden_dims: List of hidden layer sizes
            output_dim: Output dimension (1 for scalar field magnitude)
            dropout_rate: Probability of dropping connections (0.1-0.5 typical)
        """
        super(MCDropoutMagneticFieldNet, self).__init__()

        self.dropout_rate = dropout_rate

        # Build network architecture
        layers = []
        prev_dim = input_dim

        for i, hidden_dim in enumerate(hidden_dims):
            # Fully-connected layer
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            
            # Dropout after activation (except final hidden layer)
            if i < len(hidden_dims) - 1:
                layers.append(nn.Dropout(dropout_rate))
            prev_dim = hidden_dim

        # Output layer (no activation for regression)
        layers.append(nn.Linear(prev_dim, output_dim))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        """Standard forward pass (training or single inference)"""
        return self.network(x)

    def predict_with_uncertainty(self, x, n_samples=100):
        """
        Generate predictions with uncertainty estimates using Monte Carlo Dropout
        
        Args:
            x: Input tensor (batch_size, input_dim)
            n_samples: Number of stochastic forward passes (T in theory)
        
        Returns:
            Dictionary with:
            - mean: Expected prediction (epistemic uncertainty marginalized)
            - std: Standard deviation (epistemic uncertainty magnitude)
            - percentile_5/95: 90% confidence interval bounds
            - all_predictions: Full distribution for further analysis
        
        Implementation note: self.train() keeps dropout active during inference,
        which is the key to MC Dropout methodology.
        """
        self.train()  # CRITICAL: Enable dropout during inference

        predictions = []

        with torch.no_grad():  # No gradient computation needed
            for _ in range(n_samples):
                # Each forward pass uses a different random dropout mask
                pred = self(x)
                predictions.append(pred)

        # Stack predictions: (n_samples, batch_size, output_dim)
        predictions = torch.stack(predictions)

        # Calculate statistics across MC samples (dim=0)
        mean = predictions.mean(dim=0)
        std = predictions.std(dim=0)

        # Calculate percentiles for confidence intervals
        percentiles_5 = torch.quantile(predictions, 0.05, dim=0)
        percentiles_95 = torch.quantile(predictions, 0.95, dim=0)

        return {
            'mean': mean,
            'std': std,
            'percentile_5': percentiles_5,
            'percentile_95': percentiles_95,
            'all_predictions': predictions
        }

# Demonstration of MC Dropout on synthetic magnetic field data
def demonstrate_mc_dropout():
    """
    Demonstrate MC Dropout on synthetic magnetic field prediction task
    
    Simulates a scenario where:
    1. Training data has heteroscedastic noise (input-dependent variance)
    2. Model must predict field values with confidence intervals
    3. High uncertainty should correlate with high prediction errors
    
    This mimics real EM scenarios where:
    - Linear regions (low B) have low noise → low uncertainty
    - Saturation regions (high B) have high noise → high uncertainty
    """

    print(" Monte Carlo Dropout Demonstration")
    print("=" * 50)

    # Generate synthetic data simulating EM field prediction
    torch.manual_seed(42)
    n_samples = 1000

    # Input features: [geometry_param1, material_property, excitation_level]
    X = torch.randn(n_samples, 3)

    # True underlying function (nonlinear, representative of B-field physics)
    # Quadratic + interaction terms simulate magnetic saturation effects
    y_true = 0.5 * X[:, 0]**2 + 0.3 * X[:, 1] * X[:, 2] + 0.1 * torch.randn(n_samples)
    y_true = y_true.unsqueeze(1)

    # Add heteroscedastic noise (aleatoric uncertainty)
    # Noise increases with X[:, 0] (simulates higher noise in saturation)
    noise_std = 0.1 + 0.05 * torch.abs(X[:, 0])
    y = y_true + noise_std.unsqueeze(1) * torch.randn(n_samples, 1)

    # Split data: 80% train, 20% test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Create MC Dropout model
    model = MCDropoutMagneticFieldNet(
        input_dim=3, 
        hidden_dims=[32, 16],  # Compact for fast demo
        dropout_rate=0.2       # 20% dropout rate
    )

    # Training configuration
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.MSELoss()

    # Training loop
    model.train()
    epochs = 200

    print("\n Training MC Dropout model...")
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 50 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

    # Generate uncertainty estimates on test set
    print("\n Generating uncertainty estimates (T=50 MC samples)...")
    uncertainty_results = model.predict_with_uncertainty(X_test[:100], n_samples=50)

    # Calculate evaluation metrics
    mean_pred = uncertainty_results['mean']
    std_pred = uncertainty_results['std']
    true_values = y_test[:100]

    mse = torch.mean((mean_pred - true_values)**2)
    
    # Coverage: How many true values fall within 90% CI?
    coverage_90 = torch.mean(
        ((true_values >= uncertainty_results['percentile_5']) &
         (true_values <= uncertainty_results['percentile_95'])).float()
    )

    # Create comprehensive visualization
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Monte Carlo Dropout: Uncertainty Quantification Results',
                 fontsize=16, fontweight='bold')

    # Plot 1: Predictions with confidence intervals
    sample_idx = torch.arange(50)
    axes[0, 0].plot(sample_idx.numpy(), true_values[:50].numpy(), 'k-o',
                   label='True Values', markersize=4, linewidth=2)
    axes[0, 0].plot(sample_idx.numpy(), mean_pred[:50].numpy(), 'b-s',
                   label='Mean Prediction', markersize=4)
    axes[0, 0].fill_between(sample_idx.numpy(),
                           uncertainty_results['percentile_5'][:50].numpy().flatten(),
                           uncertainty_results['percentile_95'][:50].numpy().flatten(),
                           alpha=0.3, color='blue', label='90% CI')
    axes[0, 0].set_xlabel('Sample Index', fontsize=10)
    axes[0, 0].set_ylabel('Magnetic Field (normalized)', fontsize=10)
    axes[0, 0].set_title('Predictions with Confidence Intervals', fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Plot 2: Uncertainty vs Error correlation
    errors = torch.abs(mean_pred - true_values)
    axes[0, 1].scatter(std_pred.numpy().flatten(), errors.numpy().flatten(), alpha=0.6)
    axes[0, 1].set_xlabel('Predicted Uncertainty (σ)', fontsize=10)
    axes[0, 1].set_ylabel('Absolute Error |ŷ - y|', fontsize=10)
    axes[0, 1].set_title('Uncertainty vs Prediction Error', fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Add trend line
    z = np.polyfit(std_pred.numpy().flatten(), errors.numpy().flatten(), 1)
    p = np.poly1d(z)
    axes[0, 1].plot(std_pred.numpy().flatten(), 
                   p(std_pred.numpy().flatten()), 
                   "r--", alpha=0.8, label=f'Trend: y={z[0]:.2f}x+{z[1]:.2f}')
    axes[0, 1].legend()

    plt.tight_layout()
    plt.show()

    # Print quantitative results
    print(f"\n Results Summary:")
    print(f"  MSE: {mse.item():.4f}")
    print(f"  90% Coverage: {coverage_90.item():.1%} (ideal: 90%)")
    print(f"  Mean Uncertainty: {std_pred.mean().item():.4f}")
    print(f"  Uncertainty Range: [{std_pred.min().item():.4f}, {std_pred.max().item():.4f}]")

    return model, uncertainty_results

# Execute demonstration
mc_model, mc_results = demonstrate_mc_dropout()




:::note
**Interpreting MC Dropout Results**  

**Key indicators of good uncertainty estimates**:
1. **Coverage calibration**: ~90% of true values should fall within 90% CI
   - Much less → over-confident (underestimating uncertainty)
   - Much more → under-confident (overestimating uncertainty)

2. **Correlation**: High uncertainty should correlate with high error
   - Positive correlation indicates model "knows when it doesn't know"
   - Critical for safe deployment in EM design automation

3. **Spatial distribution**: In real EM problems, expect:
   - Low uncertainty in linear regions (far from saturation)
   - High uncertainty at material boundaries
   - Very high uncertainty for out-of-distribution geometries
:::

## Deep Ensembles: Robust Uncertainty Estimation

### 1. Ensemble Theory

Deep ensembles combine multiple independently trained neural networks to produce robust predictions with well-calibrated uncertainty estimates {cite}`lakshminarayanan2017simple,fort2019deep`.

**Mathematical foundation**:

**Ensemble prediction** (average over M models):
$$f_{\text{ensemble}}(x) = \frac{1}{M}\sum_{m=1}^{M} f_m(x)$$

where $f_m$ are individual models with different:
- Random weight initializations
- Training data subsets (bootstrap sampling)
- Stochastic mini-batch orderings

**Key advantage** {cite}`lakshminarayanan2017simple`: Deep ensembles provide better calibrated uncertainty than MC Dropout, especially for out-of-distribution detection—critical for electromagnetic design where novel geometries are common.

### 2. Uncertainty Decomposition: Aleatoric + Epistemic

Deep ensembles naturally capture both uncertainty types {cite}`kendall2017uncertainties,gal2017deep`:

**Total predictive variance**:

$$\text{Var}[y|x] = \underbrace{\frac{1}{M}\sum_{m=1}^{M}\sigma_m^2(x)}_{\text{Aleatoric}} + \underbrace{\frac{1}{M}\sum_{m=1}^{M}(\mu_m(x) - \bar{\mu}(x))^2}_{\text{Epistemic}}$$

where:
- $\mu_m(x)$ = mean prediction from model $m$
- $\sigma_m^2(x)$ = predicted variance from model $m$ (if heteroscedastic output)
- $\bar{\mu}(x) = \frac{1}{M}\sum_{m=1}^{M} \mu_m(x)$ = ensemble mean

**Interpretation** {cite}`hullermeier2021aleatoric`:
- **Aleatoric** (first term): Average within-model variance (data noise)
- **Epistemic** (second term): Disagreement between models (model uncertainty)

### 3. Training Strategies

**Bagging (Bootstrap Aggregating)** {cite}`breiman1996bagging`:
- Train each model on different bootstrap samples (sampling with replacement)
- Reduces variance and improves generalization
- **Implementation**: For N training samples, each model sees ~63% unique samples

**Snapshot ensembles** {cite}`huang2017snapshot`:
- Collect models from different epochs using cyclic learning rates
- Computational efficiency: single training run produces M models
- Trade-off: Less diversity than independent training

**Diverse architectures** {cite}`fort2019deep`:
- Use different network depths, widths, or activation functions
- Maximizes ensemble diversity
- **For EM**: Combine dilated and standard convolutions for complementary spatial biases

:::{important}
**Computational Trade-offs: MC Dropout vs. Deep Ensembles**  

**MC Dropout** {cite}`gal2016dropout`:
- Training: 1× network (fast)
- Inference: T forward passes (~50-100× single prediction)
- Uncertainty quality: Good, but can be under-confident
- Best for: Prototyping, resource-constrained deployment

**Deep Ensembles** {cite}`lakshminarayanan2017simple`:
- Training: M× networks (~5-10× training time)
- Inference: M forward passes (5-10× single prediction)
- Uncertainty quality: Excellent, well-calibrated for OOD detection
- Best for: Production, safety-critical EM applications

**Recommendation for EM design**:
- **Early exploration**: MC Dropout (fast prototyping)
- **Production deployment**: 5-member ensemble (reliability)
- **Critical validation**: 10-member ensemble + MC Dropout (maximum confidence)
:::

In [ ]:
# Deep Ensemble Implementation
class DeepEnsemble:
    """Deep Ensemble for magnetic field prediction"""

    def __init__(self, n_models=5, input_dim=3, hidden_dims=[32, 16], output_dim=1):
        self.n_models = n_models
        self.models = []

        for _ in range(n_models):
            model = nn.Sequential(
                nn.Linear(input_dim, hidden_dims[0]),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(hidden_dims[0], hidden_dims[1]),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(hidden_dims[1], output_dim)
            )
            self.models.append(model)

    def train_ensemble(self, X_train, y_train, epochs=200, lr=0.01):
        """Train all models in the ensemble"""
        print(f"  Training ensemble with {self.n_models} models...")

        for i, model in enumerate(self.models):
            print(f"  Training model {i+1}/{self.n_models}...")

            optimizer = torch.optim.Adam(model.parameters(), lr=lr)
            criterion = nn.MSELoss()

            # Bootstrap sampling
            indices = torch.randint(0, len(X_train), (len(X_train),))
            X_bootstrap = X_train[indices]
            y_bootstrap = y_train[indices]

            model.train()
            for epoch in range(epochs):
                optimizer.zero_grad()
                outputs = model(X_bootstrap)
                loss = criterion(outputs, y_bootstrap)
                loss.backward()
                optimizer.step()

                if (epoch + 1) % 50 == 0:
                    print(f"    Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

    def eval(self):
        """Set all models to evaluation mode"""
        for model in self.models:
            model.eval()

    def predict_with_uncertainty(self, x):
        """Generate ensemble predictions with uncertainty"""
        self.eval()

        predictions = []

        with torch.no_grad():
            for model in self.models:
                pred = model(x)
                predictions.append(pred)

        predictions = torch.stack(predictions)

        # Calculate ensemble statistics
        mean = predictions.mean(dim=0)
        std = predictions.std(dim=0)

        return {
            'mean': mean,
            'std': std,
            'all_predictions': predictions
        }

# Demonstrate Deep Ensembles
def demonstrate_deep_ensembles():
    """Demonstrate Deep Ensemble on magnetic field prediction"""

    print(" Deep Ensemble Demonstration")
    print("=" * 50)

    # Generate synthetic data
    torch.manual_seed(123)
    n_samples = 800

    # More complex function for ensemble demonstration
    X = torch.randn(n_samples, 3)
    y_true = torch.sin(X[:, 0]) * X[:, 1] + 0.5 * X[:, 2]**2 + 0.1 * torch.randn(n_samples)
    y_true = y_true.unsqueeze(1)

    # Add heteroscedastic noise
    noise_std = 0.05 + 0.1 * torch.abs(X[:, 1])
    y = y_true + noise_std.unsqueeze(1) * torch.randn(n_samples, 1)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Create and train ensemble
    ensemble = DeepEnsemble(n_models=5, input_dim=3, hidden_dims=[32, 16])
    ensemble.train_ensemble(X_train, y_train, epochs=150)

    # Generate predictions
    print("\n Generating ensemble predictions...")
    ensemble_results = ensemble.predict_with_uncertainty(X_test[:100])

    # Calculate metrics
    mean_pred = ensemble_results['mean']
    std_pred = ensemble_results['std']
    true_values = y_test[:100]

    mse = torch.mean((mean_pred - true_values)**2)

    # Create visualization
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Deep Ensemble: Uncertainty Quantification Results',
                 fontsize=16, fontweight='bold')

    # Plot 1: Ensemble mean with confidence intervals
    sample_idx = torch.arange(30)
    axes[0, 0].plot(sample_idx.numpy(), true_values[:30].numpy(), 'k-o',
                   label='True Values', markersize=4)
    axes[0, 0].plot(sample_idx.numpy(), mean_pred[:30].numpy(), 'b-s',
                   label='Ensemble Mean', markersize=4)
    axes[0, 0].fill_between(sample_idx.numpy(),
                           (mean_pred[:30] - 2*std_pred[:30]).numpy().flatten(),
                           (mean_pred[:30] + 2*std_pred[:30]).numpy().flatten(),
                           alpha=0.3, color='blue', label='95% CI')
    axes[0, 0].set_xlabel('Sample Index')
    axes[0, 0].set_ylabel('Magnetic Field')
    axes[0, 0].set_title('Ensemble Prediction with Uncertainty')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Plot 2: Uncertainty vs Error
    errors = torch.abs(mean_pred - true_values)
    axes[0, 1].scatter(std_pred.numpy().flatten(), errors.numpy().flatten(), alpha=0.6)
    axes[0, 1].set_xlabel('Ensemble Uncertainty (σ)')
    axes[0, 1].set_ylabel('Absolute Error')
    axes[0, 1].set_title('Uncertainty vs Prediction Error')
    axes[0, 1].grid(True, alpha=0.3)

    # Plot 3: Individual model predictions
    individual_predictions = ensemble_results['all_predictions']
    for i in range(min(3, individual_predictions.shape[0])):
        axes[1, 0].plot(sample_idx.numpy(), individual_predictions[i, :30].numpy(),
                       alpha=0.6, label=f'Model {i+1}')
    axes[1, 0].plot(sample_idx.numpy(), true_values[:30].numpy(), 'k-o',
                   label='True Values', markersize=4, linewidth=2)
    axes[1, 0].set_xlabel('Sample Index')
    axes[1, 0].set_ylabel('Magnetic Field')
    axes[1, 0].set_title('Individual Model Predictions')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Plot 4: Uncertainty distribution
    axes[1, 1].hist(std_pred.numpy().flatten(), bins=20, alpha=0.7, edgecolor='black')
    axes[1, 1].set_xlabel('Ensemble Uncertainty (σ)')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('Uncertainty Distribution')
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Print results
    print(f"\n Ensemble Results Summary:")
    print(f"  Ensemble MSE: {mse.item():.4f}")
    print(f"  Mean Uncertainty: {std_pred.mean().item():.4f}")
    print(f"  Uncertainty Range: [{std_pred.min().item():.4f}, {std_pred.max().item():.4f}]")

    return ensemble

# Run demonstration
ensemble_model = demonstrate_deep_ensembles()

## Summary and Implementation Guidelines

This chapter provided a comprehensive framework for uncertainty quantification in deep learning-based magnetic field prediction, addressing the critical limitation identified in Section 4d: **knowing when predictions are reliable**.

### Key Takeaways

#### 1. **Theoretical Foundation** {cite}`kendall2017uncertainties,hullermeier2021aleatoric`

- **Aleatoric uncertainty**: Irreducible data noise (FEA discretization, material tolerances)
- **Epistemic uncertainty**: Reducible model uncertainty (training data sparsity, extrapolation)
- **Decomposition**: Total uncertainty = aleatoric + epistemic (law of total variance)
- **Engineering implication**: Use epistemic uncertainty to guide active learning

#### 2. **Practical Methods Comparison**

| Method | Training Cost | Inference Cost | Calibration | OOD Detection | Best Use Case |
|--------|--------------|----------------|-------------|---------------|---------------|
| **MC Dropout** {cite}`gal2016dropout` | 1× | 50-100× | Good | Moderate | Prototyping, fast iteration |
| **Deep Ensembles** {cite}`lakshminarayanan2017simple` | 5-10× | 5-10× | Excellent | Excellent | Production, safety-critical |
| **Bayesian NNs** {cite}`mackay1992bayesian` | 10-50× | 1-10× | Excellent | Excellent | Research, maximum rigor |

**Recommendation for EM design**: Start with MC Dropout (fastest prototyping), transition to 5-member ensemble for production deployment.

#### 3. **Engineering Applications in Electromagnetic Design**

**Design automation workflow** {cite}`amodei2016concrete,varshney2016engineering`:
1. **Initial screening**: CNN with uncertainty (evaluate 1,000 designs)
2. **Uncertainty filtering**: Reject designs with epistemic uncertainty > threshold
3. **Active learning**: Run FEA on high-uncertainty designs to improve model
4. **Final validation**: FEA verification of low-uncertainty top candidates

**Safety-critical decision making** {cite}`holm2020review`:
- **Conservative estimates**: Use 95% CI upper bound for worst-case analysis
- **Regulatory compliance**: Document uncertainty in safety calculations
- **Fail-safe**: If uncertainty exceeds acceptable threshold, default to FEA

### Implementation Roadmap

**Phase 1: Prototyping** (Weeks 1-2)
1. Implement MC Dropout with dropout_rate=0.2
2. Validate coverage calibration (90% CI should capture 85-95% of test data)
3. Establish baseline performance metrics (MSE, coverage, correlation)

**Phase 2: Calibration** (Weeks 3-4) {cite}`guo2017calibration,kuleshov2018accurate`
1. Measure calibration on held-out test set
2. Apply temperature scaling if over/under-confident
3. Monitor uncertainty-error correlation

**Phase 3: Production Deployment** (Weeks 5-8)
1. Train 5-member deep ensemble with bootstrap sampling
2. Implement automated uncertainty monitoring
3. Deploy with uncertainty-aware decision rules

### Best Practices for Electromagnetic Modeling

**1. Always Validate Calibration** {cite}`guo2017calibration`:
- Plot reliability diagrams (predicted vs. observed confidence)
- Target: 90% CI should contain 85-95% of true values (±5% acceptable)
- Recalibrate if systematic bias detected

**2. Physical Consistency Checks** {cite}`sadiku2014elements`:
- Uncertainty should respect physics: higher near boundaries, saturation regions
- Zero uncertainty at known boundary conditions indicates model defect
- Validate ∇·B = 0 even in high-uncertainty regions

**3. Out-of-Distribution Detection** {cite}`amodei2016concrete`:
- High epistemic + low aleatoric → extrapolation (dangerous)
- Low epistemic + high aleatoric → noisy region (acceptable)
- Implement automated flagging for epistemic uncertainty > 2× training median

**4. Active Learning Integration** {cite}`settles2009active`:
- Use uncertainty to guide FEA sample collection
- Query simulator at max(epistemic uncertainty) locations
- Re-train periodically to reduce uncertainty in critical design regions

**5. Stakeholder Communication** {cite}`varshney2016engineering`:
- Report predictions with confidence intervals, not point estimates
- Clearly document when model is extrapolating (high epistemic uncertainty)
- Provide uncertainty-aware recommendations (e.g., "95% confident torque > 50 Nm")

### Connections to Advanced Topics

**Chapter 6 Preview: Physics-Informed Neural Networks (PINNs)**

This chapter established epistemic uncertainty quantification via dropout and ensembles. Chapter 6 explores an orthogonal approach: **reducing epistemic uncertainty by incorporating physics constraints** {cite}`raissi2019physics,karniadakis2021physics`.

**Key connection**: While uncertainty quantification identifies *when* predictions are unreliable, PINNs aim to make predictions more reliable by enforcing Maxwell's equations {cite}`sadiku2014elements`:

$$\nabla \times \mathbf{H} = \mathbf{J}, \quad \nabla \cdot \mathbf{B} = 0$$

**Synergy**: Combine both approaches—PINNs for physics-consistent predictions + MC dropout for remaining uncertainty quantification—yielding reliable predictions with trustworthy confidence intervals {cite}`yang2021adversarial`.

### Concluding Remarks

Uncertainty quantification transforms machine learning from a "black box" into a **trustworthy engineering tool** for electromagnetic design. By quantifying confidence in predictions, engineers can:
- **Automate design exploration** with safety guarantees
- **Guide active learning** for efficient data collection
- **Make risk-informed decisions** in safety-critical applications

The methods presented here—MC Dropout and Deep Ensembles—provide practical, computationally efficient uncertainty estimates suitable for industrial electromagnetic design workflows {cite}`lakshminarayanan2017simple,gal2016dropout}.